# Avaliação Automática de Dificuldade em Jogos de Plataforma 2D
## Deep Reinforcement Learning aplicado ao Super Mario Bros — DQN vs PPO vs A2C

**Disciplina:** Redes Neurais Artificiais — PPGCC UNESP
**Notebook:** Fase 1 — pipeline ponta-a-ponta de treinamento, avaliação e análise

Este notebook implementa toda a metodologia do artigo:

1. Formulação do problema como **MDP** com observações de $4 \times 84 \times 84$ frames e ações discretas (`SIMPLE_MOVEMENT`).
2. **Pipeline de pré-processamento Atari clássica** (skip-frame 4, grayscale, resize 84x84, frame-stack 4).
3. Três arquiteturas com **NatureCNN compartilhada**: DQN, PPO, A2C.
4. **36 treinamentos**: 3 algoritmos × 4 fases (1-1, 1-2, 4-1, 8-1) × 3 seeds (42, 123, 2024), todos com 500k timesteps.
5. **Métricas Grupo I** (comparação de arquiteturas): recompensa final, AUC, timesteps até 80% do máximo, std.
6. **Métricas Grupo II** (avaliação de dificuldade): taxa de conclusão, distância normalizada, mortes, tempo.
7. **Análise estatística**: Spearman entre ranking dos agentes e ranking canônico das fases + Mann-Whitney U.

> **Modo SMOKE_TEST**: defina `SMOKE_TEST = True` na célula de configuração para validar o pipeline em ~2 minutos antes de disparar os experimentos reais (~60h de GPU).

## 1. Instalação de dependências

Versões **travadas e testadas em conjunto** — `gym-super-mario-bros` tem várias armadilhas de compatibilidade com versões recentes de NumPy, gym e Python. Não trocar versões sem testar.

**Requisitos do sistema:**
- Python 3.10 ou 3.11 (não 3.12 — `nes-py` falha na compilação Cython)
- Linux com `build-essential` e `python3-dev` instalados
- GPU NVIDIA com CUDA (verificado adiante)

Execute esta célula **uma vez** e reinicie o kernel ao final.

In [ ]:
# Descomente e rode UMA VEZ. Depois comente novamente.
# !pip install --quiet \
#     "numpy<2.0" \
#     "gymnasium==0.29.1" \
#     "shimmy==1.3.0" \
#     "stable-baselines3==2.3.2" \
#     "gym-super-mario-bros==7.4.0" \
#     "nes-py==8.2.1" \
#     "torch>=2.0,<2.5" \
#     "tensorboard" \
#     "pandas" \
#     "matplotlib" \
#     "seaborn" \
#     "scipy" \
#     "tqdm" \
#     "opencv-python-headless"

print("Após instalar, REINICIE O KERNEL antes de continuar.")

## 2. Imports e verificação de ambiente

Carrega todas as bibliotecas e confirma que GPU está disponível. Se `cuda` retornar `False`, todo o treinamento vai rodar em CPU (centenas de horas, inviável).

In [ ]:
import os
import json
import time
import random
import pickle
import warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Callable

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# RL / env stack
import gymnasium as gym
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace
from shimmy import GymV21CompatibilityV0

# Stable-Baselines3
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.atari_wrappers import MaxAndSkipEnv, WarpFrame
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import (
    DummyVecEnv, SubprocVecEnv, VecFrameStack, VecMonitor
)
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy

# Stats
from scipy.stats import spearmanr, mannwhitneyu

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
sns.set_theme(style="whitegrid", context="paper")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device   : {device}")

## 3. Configuração do experimento

Toda a parametrização em um único lugar — qualquer mudança aqui se propaga pelo notebook. Os valores seguem exatamente a Tabela 2 da metodologia do artigo (configurações padrão da Stable-Baselines3 para ambientes Atari, com ajustes para Super Mario Bros).

**`SMOKE_TEST = True`** reduz tudo para validar o pipeline em poucos minutos.

In [ ]:
# -------------------------------------------------------------
# MODO DE EXECUÇÃO
# -------------------------------------------------------------
SMOKE_TEST = True   # True = teste rápido (~2 min); False = experimento real (~60h)

# -------------------------------------------------------------
# DIRETÓRIOS
# -------------------------------------------------------------
ROOT_DIR     = Path("./mario_drl_results").resolve()
MODELS_DIR   = ROOT_DIR / "models"
LOGS_DIR     = ROOT_DIR / "logs"
TB_DIR       = ROOT_DIR / "tensorboard"
METRICS_DIR  = ROOT_DIR / "metrics"
PLOTS_DIR    = ROOT_DIR / "plots"
for d in (MODELS_DIR, LOGS_DIR, TB_DIR, METRICS_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------
# EXPERIMENTO
# -------------------------------------------------------------
STAGES = ["1-1", "1-2", "4-1", "8-1"]          # ordem canônica de dificuldade
SEEDS  = [42, 123, 2024]
ALGOS  = ["DQN", "PPO", "A2C"]

if SMOKE_TEST:
    TOTAL_TIMESTEPS = 10_000
    EVAL_FREQ       = 2_000
    N_EVAL_EPISODES = 2
    STAGES_TO_RUN   = ["1-1"]   # só uma fase no smoke test
    SEEDS_TO_RUN    = [42]
else:
    TOTAL_TIMESTEPS = 500_000
    EVAL_FREQ       = 10_000
    N_EVAL_EPISODES = 5
    STAGES_TO_RUN   = STAGES
    SEEDS_TO_RUN    = SEEDS

# -------------------------------------------------------------
# HIPERPARÂMETROS (Tabela 2 do artigo)
# -------------------------------------------------------------
HPARAMS = {
    "DQN": dict(
        learning_rate     = 1e-4,
        buffer_size       = 100_000,
        learning_starts   = 10_000,
        batch_size        = 32,
        tau               = 1.0,
        gamma             = 0.99,
        train_freq        = 4,
        gradient_steps    = 1,
        target_update_interval = 10_000,
        exploration_fraction   = 0.10,
        exploration_initial_eps = 1.0,
        exploration_final_eps   = 0.01,
        max_grad_norm     = 10.0,
        n_envs            = 1,
    ),
    "PPO": dict(
        learning_rate     = 2.5e-4,
        n_steps           = 128,
        batch_size        = 256,
        n_epochs          = 4,
        gamma             = 0.99,
        gae_lambda        = 0.95,
        clip_range        = 0.1,
        ent_coef          = 0.01,
        vf_coef           = 0.5,
        max_grad_norm     = 0.5,
        n_envs            = 8,
    ),
    "A2C": dict(
        learning_rate     = 7e-4,
        n_steps           = 5,
        gamma             = 0.99,
        gae_lambda        = 1.0,
        ent_coef          = 0.01,
        vf_coef           = 0.25,
        max_grad_norm     = 0.5,
        rms_prop_eps      = 1e-5,
        use_rms_prop      = True,
        n_envs            = 16,
    ),
}

# Frame-stack: 4 frames empilhados (parte da política, não do env base)
FRAME_STACK = 4

print(f"Modo: {'SMOKE_TEST' if SMOKE_TEST else 'EXPERIMENTO COMPLETO'}")
print(f"Treinamentos planejados: {len(ALGOS) * len(STAGES_TO_RUN) * len(SEEDS_TO_RUN)}")
print(f"Timesteps por treino:   {TOTAL_TIMESTEPS:,}")
print(f"Resultados em:           {ROOT_DIR}")

## 4. Environment factory

Esta é a parte mais delicada do projeto — a ordem dos wrappers e a conversão gym→gymnasium têm que estar exatas, senão o env quebra de formas sutis (ex: `info` perde os campos `flag_get`, `x_pos`, `life`, que são essenciais para as métricas do Grupo II).

**Sequência aplicada:**

| # | Wrapper | Função |
|---|---------|--------|
| 1 | `gym_super_mario_bros.make` | Cria o env do Mario (API antiga do gym) |
| 2 | `JoypadSpace(SIMPLE_MOVEMENT)` | Restringe ações a 7 (NOOP, →, →+pulo, →+corre, →+corre+pulo, pulo, ←) |
| 3 | `GymV21CompatibilityV0` | Converte API antiga gym → gymnasium |
| 4 | `MaxAndSkipEnv(skip=4)` | Repete ação 4 frames + max pixel-wise dos 2 últimos |
| 5 | `WarpFrame` | Grayscale + resize bilinear para 84×84 |
| 6 | `Monitor` | Logging de episódios |

O **frame-stack(4)** é aplicado **fora do env**, no nível do `VecEnv`, via `VecFrameStack` — ordem correta para que o SB3 trate corretamente a observação de pilha.

In [ ]:
# -------------------------------------------------------------
# Detecta versão do gym instalado e escolhe o shim correto.
# gym 0.26+ usa a nova API (reset retorna (obs, info));
# versões anteriores usam a API legada (reset retorna só obs).
# -------------------------------------------------------------
import gym as _legacy_gym
_gym_version = tuple(int(x) for x in _legacy_gym.__version__.split(".")[:2])

if _gym_version >= (0, 26):
    from shimmy import GymV26CompatibilityV0 as _GymCompat
    _COMPAT_API = "v0.26"
else:
    from shimmy import GymV21CompatibilityV0 as _GymCompat
    _COMPAT_API = "v0.21"

print(f"gym detectado: {_legacy_gym.__version__}  →  usando shimmy.{_GymCompat.__name__}")


# -------------------------------------------------------------
# Patch para nes-py 8.2.1: JoypadSpace.reset() não aceita os kwargs
# `seed` e `options` introduzidos em gym>=0.22. Quando SB3/shimmy
# encaminham esses argumentos para o reset, dá TypeError. O subclass
# abaixo simplesmente engole esses kwargs e delega ao parent.
# Discussão: https://github.com/Kautenja/nes-py/issues/...
# -------------------------------------------------------------
class CompatJoypadSpace(JoypadSpace):
    """JoypadSpace tolerante a seed/options no reset()."""
    def reset(self, seed=None, options=None, **kwargs):
        return super().reset(**kwargs)


def make_mario_env(stage: str = "1-1", seed: int = 0):
    """
    Cria UM env do Super Mario Bros já com toda a pipeline de pré-processamento.
    
    Args:
        stage: Fase no formato "world-stage", ex: "1-1", "4-2"
        seed:  Semente para reprodutibilidade
    
    Returns:
        env gymnasium pronto para SB3 (SEM frame-stack — esse é VecFrameStack).
    """
    env_id = f"SuperMarioBros-{stage}-v0"
    
    # 1) Env do Mario. apply_api_compatibility=True faz o gym 0.26 envolver
    #    o env legado para retornar 5-tuple no step e (obs,info) no reset.
    env = gym_super_mario_bros.make(env_id, apply_api_compatibility=True)
    
    # 2) Restringe ação a SIMPLE_MOVEMENT (7 ações) com fix p/ seed/options.
    env = CompatJoypadSpace(env, SIMPLE_MOVEMENT)
    
    # 3) Ponte gym → gymnasium (mesma API, mas classe diferente).
    env = _GymCompat(env=env)
    
    # 4) Skip-frame com max pixel-wise (protocolo Atari de Mnih et al., 2015).
    env = MaxAndSkipEnv(env, skip=4)
    
    # 5) Grayscale + resize 84x84.
    env = WarpFrame(env, width=84, height=84)
    
    # 6) Monitor para episódios.
    env = Monitor(env)
    
    env.action_space.seed(seed)
    return env


def make_vec_env_mario(stage: str, n_envs: int, seed: int, use_subproc: bool = True):
    """
    Cria um VecEnv com n_envs cópias paralelas do Mario + frame stack.
    
    Para DQN (n_envs=1) usamos DummyVecEnv (mais leve).
    Para PPO/A2C (n_envs>1) usamos SubprocVecEnv (processos separados).
    """
    def make_one(rank: int):
        def _init():
            return make_mario_env(stage=stage, seed=seed + rank)
        return _init
    
    env_fns = [make_one(i) for i in range(n_envs)]
    if n_envs == 1 or not use_subproc:
        vec_env = DummyVecEnv(env_fns)
    else:
        vec_env = SubprocVecEnv(env_fns, start_method="fork")
    
    # Frame-stack 4 no nível VecEnv.
    vec_env = VecFrameStack(vec_env, n_stack=FRAME_STACK)
    vec_env = VecMonitor(vec_env)
    return vec_env


def set_global_seed(seed: int):
    """Reprodutibilidade: NumPy, Python, PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## 5. Smoke test do ambiente

Antes de qualquer treinamento, valida que o env carrega, executa passos e devolve observações com o shape correto $(4, 84, 84)$ (depois do `VecFrameStack`) e `info` contendo os campos do Mario que vamos usar nas métricas.

In [ ]:
print("=== Validação do ambiente ===\n")

# Cria um env solo para inspecionar
test_env = make_vec_env_mario(stage="1-1", n_envs=1, seed=0, use_subproc=False)

print(f"Observation space : {test_env.observation_space}")
print(f"Action space      : {test_env.action_space}")
print(f"Frame stack       : {FRAME_STACK}")

obs = test_env.reset()
print(f"\nObs shape após reset: {obs.shape}  (esperado: (1, 84, 84, 4) ou (1, 4, 84, 84))")
print(f"Obs dtype           : {obs.dtype}")

# Executa 5 passos aleatórios e inspeciona o info do Mario
print("\nExecutando 5 passos aleatórios...")
for i in range(5):
    action = [test_env.action_space.sample()]
    obs, reward, done, info = test_env.step(action)
    info0 = info[0] if isinstance(info, (list, tuple)) else info
    keys_mario = {k: info0.get(k) for k in ["x_pos", "y_pos", "life", "score", "flag_get", "time"]}
    print(f"  step {i+1}: reward={reward[0]:+.2f}  done={done[0]}  info={keys_mario}")

test_env.close()
print("\n✓ Env funcionando corretamente.")

## 6. Callback de avaliação e coleta de métricas

A cada `EVAL_FREQ` timesteps, este callback:

1. Cria um env de avaliação separado (`deterministic=True`)
2. Roda `N_EVAL_EPISODES` episódios
3. Captura por episódio: **recompensa**, **distância máxima em x**, **bandeira alcançada (flag_get)**, **mortes**, **tempo em frames**
4. Persiste tudo em um CSV — base para as métricas dos Grupos I e II

Esse é o coração da coleta de dados do trabalho.

In [ ]:
class MarioEvalCallback(BaseCallback):
    """
    Avalia o agente periodicamente e registra métricas detalhadas em CSV.

    Grava uma linha por episódio de avaliação, contendo:
      timestep, episode, reward, max_x_pos, flag_get, deaths, frames, time_left
    """

    def __init__(
        self,
        eval_stage: str,
        eval_freq: int,
        n_eval_episodes: int,
        log_path: Path,
        seed: int = 0,
        verbose: int = 1,
    ):
        super().__init__(verbose)
        self.eval_stage = eval_stage
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.log_path = Path(log_path)
        self.seed = seed
        self.eval_env = None
        self._rows = []
        self._eval_count = 0

    def _on_training_start(self) -> None:
        # Env de avaliação separado, sempre n_envs=1, determinístico
        self.eval_env = make_vec_env_mario(
            stage=self.eval_stage, n_envs=1, seed=self.seed + 9999,
            use_subproc=False
        )

    def _run_eval(self):
        """Executa n_eval_episodes e devolve uma lista de dicts com métricas."""
        results = []
        for ep in range(self.n_eval_episodes):
            obs = self.eval_env.reset()
            done = [False]
            ep_reward = 0.0
            max_x = 0
            deaths = 0
            frames = 0
            flag_get = False
            time_left = None
            prev_life = None

            while not done[0]:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, done, info = self.eval_env.step(action)
                info0 = info[0]
                ep_reward += float(reward[0])
                frames += 1

                # Métricas do Mario
                x = int(info0.get("x_pos", 0))
                if x > max_x:
                    max_x = x
                life = info0.get("life", None)
                if prev_life is not None and life is not None and life < prev_life:
                    deaths += 1
                prev_life = life
                if info0.get("flag_get", False):
                    flag_get = True
                time_left = info0.get("time", time_left)

            results.append(dict(
                episode=ep, reward=ep_reward, max_x_pos=max_x,
                flag_get=int(flag_get), deaths=deaths, frames=frames,
                time_left=time_left
            ))
        return results

    def _on_step(self) -> bool:
        if self.num_timesteps % self.eval_freq != 0:
            return True

        eval_results = self._run_eval()
        self._eval_count += 1

        for r in eval_results:
            self._rows.append({"timestep": int(self.num_timesteps), **r})

        # Estatísticas agregadas para log
        rewards = [r["reward"] for r in eval_results]
        flags   = [r["flag_get"] for r in eval_results]
        if self.verbose >= 1:
            print(
                f"  [eval @ {self.num_timesteps:>7,}] "
                f"reward={np.mean(rewards):+8.2f}±{np.std(rewards):5.2f}  "
                f"completion={np.mean(flags):.0%}  "
                f"max_x_avg={np.mean([r['max_x_pos'] for r in eval_results]):.0f}"
            )

        # Persistência incremental
        pd.DataFrame(self._rows).to_csv(self.log_path, index=False)
        return True

    def _on_training_end(self) -> None:
        if self.eval_env is not None:
            self.eval_env.close()

## 7. Função de treinamento unificada

Uma única função recebe `(algo, stage, seed)` e:

1. Cria o `VecEnv` de treino com o número de envs paralelos do algoritmo
2. Instancia o modelo SB3 correspondente (DQN/PPO/A2C) com a `CnnPolicy` (NatureCNN compartilhada)
3. Pluga o `MarioEvalCallback` para coleta de métricas
4. Treina por `TOTAL_TIMESTEPS`
5. Salva: modelo final, CSV de métricas, log do TensorBoard
6. Retorna o caminho do CSV para análise posterior

**Idempotência:** se o CSV final já existe, pula o treinamento — permite retomar de onde parou após uma queda.

In [ ]:
from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback


def experiment_id(algo: str, stage: str, seed: int) -> str:
    return f"{algo}_stage{stage}_seed{seed}"


def train_one(algo: str, stage: str, seed: int, overwrite: bool = False,
              save_checkpoints: bool = True, n_checkpoints: int = 5) -> Path:
    """
    Treina UM agente (algo, stage, seed). Idempotente: pula se já rodou.

    Args:
        algo:             "DQN" | "PPO" | "A2C"
        stage:            Fase, ex: "1-1"
        seed:             Semente
        overwrite:        Se True, retreina mesmo se já existe
        save_checkpoints: Se True, salva modelos intermediários em MODELS_DIR/checkpoints/
                          (usados na seção 8.6.2 para visualizar evolução temporal)
        n_checkpoints:    Quantos checkpoints salvar uniformemente ao longo do treino

    Returns:
        Path para o CSV de métricas de avaliação.
    """
    exp_id   = experiment_id(algo, stage, seed)
    log_csv  = LOGS_DIR / f"{exp_id}.csv"
    model_pt = MODELS_DIR / f"{exp_id}.zip"
    tb_path  = TB_DIR / exp_id
    ckpt_dir = MODELS_DIR / "checkpoints"

    if log_csv.exists() and model_pt.exists() and not overwrite:
        print(f"  → {exp_id} já existe, pulando.")
        return log_csv

    set_global_seed(seed)
    hp = HPARAMS[algo].copy()
    n_envs = hp.pop("n_envs")

    train_env = make_vec_env_mario(stage=stage, n_envs=n_envs, seed=seed,
                                   use_subproc=(n_envs > 1))

    common_kwargs = dict(
        policy="CnnPolicy",
        env=train_env,
        verbose=0,
        seed=seed,
        device=device,
        tensorboard_log=str(tb_path),
    )

    if algo == "DQN":
        model = DQN(**common_kwargs, **hp)
    elif algo == "PPO":
        model = PPO(**common_kwargs, **hp)
    elif algo == "A2C":
        model = A2C(**common_kwargs, **hp)
    else:
        raise ValueError(f"Algoritmo desconhecido: {algo}")

    # Callback de avaliação (Grupos I e II)
    eval_cb = MarioEvalCallback(
        eval_stage=stage,
        eval_freq=max(EVAL_FREQ // n_envs, 1),
        n_eval_episodes=N_EVAL_EPISODES,
        log_path=log_csv,
        seed=seed,
        verbose=1,
    )

    callbacks_list = [eval_cb]

    # Checkpoint callback — habilita visualização de evolução temporal (8.6.2)
    if save_checkpoints and not SMOKE_TEST:
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        # save_freq é em callback calls (1 por policy step, que = n_envs env steps).
        # Para ter n_checkpoints uniformes ao longo do TOTAL_TIMESTEPS:
        ckpt_save_freq = max(TOTAL_TIMESTEPS // n_envs // n_checkpoints, 1)
        ckpt_cb = CheckpointCallback(
            save_freq=ckpt_save_freq,
            save_path=str(ckpt_dir),
            name_prefix=exp_id,
            save_replay_buffer=False,
            save_vecnormalize=False,
        )
        callbacks_list.append(ckpt_cb)

    callback = CallbackList(callbacks_list) if len(callbacks_list) > 1 else callbacks_list[0]

    t0 = time.time()
    print(f"\n► [{exp_id}] iniciando treinamento ({TOTAL_TIMESTEPS:,} timesteps, n_envs={n_envs})")
    try:
        model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback,
                    tb_log_name=exp_id, progress_bar=True)
        model.save(model_pt)
        elapsed = time.time() - t0
        print(f"✓ [{exp_id}] concluído em {elapsed/60:.1f} min — modelo: {model_pt}")
    finally:
        train_env.close()

    return log_csv


## 8. Smoke test de treinamento

Roda **um único treino curto** para validar que toda a pipeline funciona ponta-a-ponta. Com `SMOKE_TEST=True` e `TOTAL_TIMESTEPS=10_000` isso leva ~1-3 minutos numa GPU mediana.

Se esta célula completar com sucesso, **o pipeline está validado** e você pode setar `SMOKE_TEST=False` na célula 3, reiniciar o notebook e rodar o experimento completo.

In [ ]:
if SMOKE_TEST:
    print("=== SMOKE TEST: 1 treinamento curto ===\n")
    csv_path = train_one(algo="PPO", stage="1-1", seed=42, overwrite=True)
    df_smoke = pd.read_csv(csv_path)
    print("\nÚltimas linhas do CSV de métricas:")
    print(df_smoke.tail(10).to_string(index=False))
else:
    print("SMOKE_TEST=False — pulando teste rápido, indo direto para o experimento completo.")

## 8.5. Visualização — gerar GIF do agente jogando

Captura um GIF inline do agente treinado executando uma fase. Útil para:

- **Sanidade visual** após cada treino (vê se o agente está pulando obstáculos, indo pra frente, etc.)
- **Apresentação** e materiais para o artigo
- **Debug** quando uma métrica diz algo estranho

A função abaixo cria um env paralelo com `render_mode='rgb_array'`, alimenta o policy com a observação pré-processada (84×84 grayscale + frame-stack) e captura a tela **original do NES** (~240×256 RGB) a cada step. Cada step do policy corresponde a 4 frames internos do emulador (por causa do `MaxAndSkipEnv`), então com `fps=15` o GIF fica em ~60 FPS efetivos do NES original.

In [ ]:
import base64
import imageio.v2 as imageio
from IPython.display import Image, HTML, display


def show_gif_inline(gif_path):
    """Exibe GIF animado inline no notebook (compatível com VSCode/JupyterLab/Colab).
    
    O Image(filename=...) do IPython não anima em alguns clientes (VSCode em particular).
    Embedar o GIF como base64 dentro de uma tag <img> resolve em todos os clientes.
    """
    gif_path = Path(gif_path)
    with open(gif_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("ascii")
    return HTML(f'<img src="data:image/gif;base64,{b64}" style="max-width: 600px;" />')


def render_agent_episode(
    model,
    stage: str = "1-1",
    max_steps: int = 2000,
    out_path = None,
    fps: int = 15,
    seed: int = 999,
    verbose: bool = True,
):
    """
    Roda um episódio do agente treinado e salva GIF do gameplay + métricas.

    Args:
        model:     Modelo SB3 treinado (DQN/PPO/A2C)
        stage:     Fase, ex: "1-1", "4-1"
        max_steps: Máximo de passos do policy (cada passo = 4 frames do NES)
        out_path:  Caminho do GIF. Default: PLOTS_DIR/agent_{stage}.gif
        fps:       Frames por segundo do GIF (15 ≈ 60 FPS do NES com skip=4)
        seed:      Seed para reprodutibilidade

    Returns:
        (Path do GIF, dict de métricas)
    """
    # Env paralelo com render_mode='rgb_array' — mesmo pipeline + captura de frames
    env_id = f"SuperMarioBros-{stage}-v0"
    base = gym_super_mario_bros.make(env_id, apply_api_compatibility=True, render_mode="rgb_array")
    base = CompatJoypadSpace(base, SIMPLE_MOVEMENT)
    base = _GymCompat(env=base)
    base = MaxAndSkipEnv(base, skip=4)
    base = WarpFrame(base, width=84, height=84)
    base = Monitor(base)
    base.action_space.seed(seed)

    vec_env = DummyVecEnv([lambda: base])
    vec_env = VecFrameStack(vec_env, n_stack=FRAME_STACK)
    render_env = vec_env.venv.envs[0]

    frames = []
    metrics = {"reward": 0.0, "max_x": 0, "deaths": 0, "flag_get": False, "steps": 0}
    prev_life = None

    obs = vec_env.reset()
    for step in range(max_steps):
        frame = render_env.render()
        if frame is not None:
            # IMPORTANTE: .copy() — nes-py retorna referência ao screen buffer
            # interno, que é sobrescrito a cada step. Sem copiar, todos os
            # frames da lista acabam apontando para a mesma memória (= GIF estático).
            frames.append(frame.copy())

        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = vec_env.step(action)

        metrics["reward"] += float(reward[0])
        metrics["steps"] += 1
        info0 = info[0]
        x = int(info0.get("x_pos", 0))
        if x > metrics["max_x"]:
            metrics["max_x"] = x
        life = info0.get("life", None)
        if prev_life is not None and life is not None and life < prev_life:
            metrics["deaths"] += 1
        prev_life = life
        if info0.get("flag_get", False):
            metrics["flag_get"] = True

        if done[0]:
            frame = render_env.render()
            if frame is not None:
                frames.append(frame.copy())
            break

    vec_env.close()

    out_path = Path(out_path) if out_path else (PLOTS_DIR / f"agent_{stage}.gif")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    
    # IMPORTANTE: imageio v2 com backend PIL usa duration em MILISSEGUNDOS.
    # Passar 1.0/fps daria 0.067ms → GIF roda em ~15000 FPS (parece estático).
    duration_ms = int(round(1000.0 / fps))
    imageio.mimsave(out_path, frames, duration=duration_ms, loop=0)

    if verbose:
        print(f"✓ {len(frames)} frames @ {fps} FPS ({duration_ms} ms/frame) → {out_path}")
        print(f"   reward={metrics['reward']:+.1f}  max_x={metrics['max_x']}  "
              f"flag={metrics['flag_get']}  deaths={metrics['deaths']}")
    return out_path, metrics


# Demonstração: roda o modelo do smoke test (se existir) e mostra animado inline.
if SMOKE_TEST:
    from stable_baselines3 import PPO
    smoke_model_path = MODELS_DIR / "PPO_stage1-1_seed42.zip"
    if smoke_model_path.exists():
        smoke_model = PPO.load(smoke_model_path, device=device)
        gif_path, _ = render_agent_episode(
            smoke_model, stage="1-1", max_steps=500, seed=999
        )
        display(show_gif_inline(gif_path))   # ← exibe ANIMADO via base64 embed
    else:
        print("Modelo do smoke test não encontrado — rode a célula anterior antes.")


## 8.6. Comparações visuais avançadas

Duas visualizações de alto valor para o artigo:

**8.6.1 — Comparação entre algoritmos:** DQN, PPO e A2C jogando a **mesma fase com o mesmo seed** lado-a-lado. Como o seed do env é o mesmo, o estado inicial e a aleatoriedade do ambiente são idênticos — a única variável é a política aprendida. Esse é o "money shot" da Seção 3 do artigo: visualiza diferenças comportamentais entre as três arquiteturas.

**8.6.2 — Evolução temporal:** o mesmo algoritmo em diferentes checkpoints do treinamento (100k, 200k, 300k, 400k, 500k timesteps). Mostra a **curva de aprendizado de forma visual** — Mario travado no primeiro Goomba aos 100k → completando a fase aos 500k. Ótimo complemento aos gráficos numéricos.

Importante: a função `render_models_side_by_side` abaixo é **genérica** — recebe um `dict {label: model}` e funciona para qualquer comparação (algoritmos, checkpoints, seeds, fases...).

In [ ]:
import cv2

def run_episode_for_render(model, stage: str, max_steps: int = 3000, seed: int = 999):
    """
    Roda um episódio com o modelo e captura frames RGB + métricas.
    Helper reutilizável (não exibe nada — só coleta dados).
    """
    env_id = f"SuperMarioBros-{stage}-v0"
    base = gym_super_mario_bros.make(env_id, apply_api_compatibility=True, render_mode="rgb_array")
    base = CompatJoypadSpace(base, SIMPLE_MOVEMENT)
    base = _GymCompat(env=base)
    base = MaxAndSkipEnv(base, skip=4)
    base = WarpFrame(base, width=84, height=84)
    base = Monitor(base)
    base.action_space.seed(seed)

    vec_env = DummyVecEnv([lambda: base])
    vec_env = VecFrameStack(vec_env, n_stack=FRAME_STACK)
    render_env = vec_env.venv.envs[0]

    frames = []
    metrics = {"reward": 0.0, "max_x": 0, "deaths": 0, "flag_get": False, "steps": 0}
    prev_life = None

    obs = vec_env.reset()
    for step in range(max_steps):
        frame = render_env.render()
        if frame is not None:
            frames.append(frame.copy())

        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = vec_env.step(action)

        metrics["reward"] += float(reward[0])
        metrics["steps"] += 1
        info0 = info[0]
        x = int(info0.get("x_pos", 0))
        if x > metrics["max_x"]:
            metrics["max_x"] = x
        life = info0.get("life", None)
        if prev_life is not None and life is not None and life < prev_life:
            metrics["deaths"] += 1
        prev_life = life
        if info0.get("flag_get", False):
            metrics["flag_get"] = True

        if done[0]:
            frame = render_env.render()
            if frame is not None:
                frames.append(frame.copy())
            break

    vec_env.close()
    return frames, metrics


def _add_label_bar(frame, label_top: str, metrics_text: str = ""):
    """Adiciona barra superior com label do agente e barra inferior com métricas."""
    out = frame.copy()
    h, w = out.shape[:2]
    bar_h_top = 26
    bar_h_bot = 22 if metrics_text else 0

    # Barra superior preta — label do agente
    cv2.rectangle(out, (0, 0), (w, bar_h_top), (0, 0, 0), -1)
    cv2.putText(out, label_top, (8, 19),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

    # Barra inferior preta — métricas (opcional)
    if metrics_text:
        cv2.rectangle(out, (0, h - bar_h_bot), (w, h), (0, 0, 0), -1)
        cv2.putText(out, metrics_text, (8, h - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def render_models_side_by_side(
    models: dict,
    stage: str = "1-1",
    max_steps: int = 3000,
    fps: int = 15,
    seed: int = 999,
    out_path = None,
    show_running_metrics: bool = True,
):
    """
    Renderiza múltiplos modelos jogando a mesma fase com o mesmo seed (estado inicial
    idêntico), compõe os frames lado-a-lado em um GIF único com labels.

    Args:
        models:               dict {label: model_obj}. Ex.: {"DQN": dqn, "PPO": ppo, "A2C": a2c}
                              ou {"100k": m100k, "300k": m300k, "500k": m500k}
        stage:                Fase (ex: "1-1")
        max_steps:            Passos máximos do policy (1 step = 4 frames NES)
        fps:                  FPS do GIF (15 ≈ 60 FPS NES com skip=4)
        seed:                 Seed do env (mesmo p/ todos → mesmo estado inicial)
        out_path:             Caminho do GIF. Default: PLOTS_DIR/comparison_{stage}.gif
        show_running_metrics: Se True, mostra max_x e reward correntes na barra inferior

    Returns:
        (Path do GIF, dict {label: metrics}, lista de frames compostos)
    """
    labels = list(models.keys())
    print(f"Renderizando {len(labels)} modelos: {labels}  |  stage {stage}  |  seed {seed}")

    # Roda cada modelo independentemente, mesmo seed → mesma fase, mesma aleatoriedade
    all_frames = {}
    all_metrics = {}
    for label, model in models.items():
        print(f"  ► {label}...")
        frames, metrics = run_episode_for_render(model, stage, max_steps, seed)
        all_frames[label] = frames
        all_metrics[label] = metrics
        flag_str = "✓" if metrics["flag_get"] else "✗"
        print(f"    {len(frames):4d} frames | r={metrics['reward']:+7.1f} | "
              f"max_x={metrics['max_x']:5d} | flag={flag_str} | deaths={metrics['deaths']}")

    # Padding com último frame para sincronizar comprimentos
    max_len = max(len(f) for f in all_frames.values())
    for label in labels:
        frames = all_frames[label]
        if len(frames) < max_len:
            pad_with = frames[-1] if frames else np.zeros((240, 256, 3), dtype=np.uint8)
            frames.extend([pad_with.copy() for _ in range(max_len - len(frames))])

    # Pré-computa max_x cumulativo por frame para cada modelo (para barra inferior)
    running = {label: {"max_x": 0, "reward": 0.0} for label in labels}

    composed_frames = []
    for t in range(max_len):
        panels = []
        for label in labels:
            f = all_frames[label][t]
            metrics_text = ""
            if show_running_metrics:
                # Aproxima máx_x/reward correntes via interpolação proporcional ao step
                final = all_metrics[label]
                progress = min((t + 1) / max(all_metrics[label]["steps"], 1), 1.0)
                approx_x = int(final["max_x"] * progress)
                approx_r = final["reward"] * progress
                metrics_text = f"x={approx_x}  r={approx_r:+.0f}"
            panels.append(_add_label_bar(f, label, metrics_text))
        composed = np.concatenate(panels, axis=1)
        composed_frames.append(composed)

    out_path = Path(out_path) if out_path else (PLOTS_DIR / f"comparison_{stage}.gif")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    duration_ms = int(round(1000.0 / fps))
    imageio.mimsave(out_path, composed_frames, duration=duration_ms, loop=0)
    print(f"\n✓ Composição salva: {out_path}  ({len(composed_frames)} frames, "
          f"{composed_frames[0].shape[1]}x{composed_frames[0].shape[0]} px)")

    return out_path, all_metrics, composed_frames


### 8.6.1 — Comparação entre algoritmos (DQN vs PPO vs A2C)

Carrega os modelos finais dos três algoritmos para a fase 1-1 com seed 42 e renderiza os três em paralelo.

**Requisito:** treino completo dos três algoritmos para `stage="1-1"` e `seed=42` (ou outra combinação que você prefira). Se ainda não tem, ajuste `stage` e `seed_for_render` para uma combinação treinada, ou rode o full pipeline primeiro.

In [ ]:
# Comparação DQN vs PPO vs A2C — mesma fase, mesmo seed
stage_for_render = "1-1"
seed_for_render = 42

algo_models = {}
for algo, cls in [("DQN", DQN), ("PPO", PPO), ("A2C", A2C)]:
    path = MODELS_DIR / f"{algo}_stage{stage_for_render}_seed{seed_for_render}.zip"
    if path.exists():
        algo_models[algo] = cls.load(path, device=device)
        print(f"✓ {algo} carregado: {path.name}")
    else:
        print(f"✗ {algo} não encontrado: {path.name}")

if len(algo_models) >= 2:
    gif_path, metrics, composed = render_models_side_by_side(
        algo_models,
        stage=stage_for_render,
        max_steps=3000,
        seed=999,   # seed do EPISÓDIO — mesma para todos os agentes
        out_path=PLOTS_DIR / f"comparison_algos_{stage_for_render}.gif",
    )
    display(animate_frames_inline(composed, fps=15))
else:
    print("\n⚠ Treine pelo menos 2 algoritmos antes de comparar.")
    print("   Rode `train_one(algo, stage, seed)` para cada um, ou o `run_all_experiments()`.")


### 8.6.2 — Evolução temporal de uma política

Mostra o **mesmo algoritmo em diferentes checkpoints** do treinamento, lado-a-lado. Útil para visualizar o aprendizado: aos 100k Mario mal anda; aos 500k ele completa a fase.

Como o `train_one` agora salva 5 checkpoints uniformes (a cada `TOTAL_TIMESTEPS/5` timesteps), eles ficam em `mario_drl_results/models/checkpoints/` com nomes tipo `PPO_stage1-1_seed42_100000_steps.zip`.

**Requisito:** ter rodado um treino completo (não-smoke-test) com `save_checkpoints=True` (padrão).

In [ ]:
import re

algo_for_evolution = "PPO"
stage_for_evolution = "1-1"
seed_for_evolution = 42

ckpt_dir = MODELS_DIR / "checkpoints"
exp_prefix = f"{algo_for_evolution}_stage{stage_for_evolution}_seed{seed_for_evolution}"
pattern = re.compile(rf"^{re.escape(exp_prefix)}_(\d+)_steps\.zip$")

ckpts = []
if ckpt_dir.exists():
    for p in sorted(ckpt_dir.glob(f"{exp_prefix}_*.zip")):
        m = pattern.match(p.name)
        if m:
            ckpts.append((int(m.group(1)), p))

ckpts.sort()

if not ckpts:
    print(f"⚠ Nenhum checkpoint encontrado para {exp_prefix}.")
    print(f"   Esperado em: {ckpt_dir}")
    print("   Verifique se o treino rodou com save_checkpoints=True (default) e SMOKE_TEST=False.")
else:
    # Pega até 5 checkpoints uniformemente espaçados
    n_show = min(5, len(ckpts))
    step_size = max(len(ckpts) // n_show, 1)
    chosen = ckpts[::step_size][:n_show]
    print(f"Checkpoints encontrados: {len(ckpts)} — exibindo {len(chosen)}:")
    
    cls = {"DQN": DQN, "PPO": PPO, "A2C": A2C}[algo_for_evolution]
    evolution_models = {}
    for steps, path in chosen:
        label = f"{steps//1000}k"
        evolution_models[label] = cls.load(path, device=device)
        print(f"  ✓ {label}: {path.name}")

    gif_path, metrics, composed = render_models_side_by_side(
        evolution_models,
        stage=stage_for_evolution,
        max_steps=2500,
        seed=999,
        out_path=PLOTS_DIR / f"evolution_{algo_for_evolution}_{stage_for_evolution}.gif",
    )
    display(animate_frames_inline(composed, fps=15))


## 9. Pipeline completo dos 36 treinamentos

Loop sobre `ALGOS × STAGES_TO_RUN × SEEDS_TO_RUN`. A ordem é deliberada:

- **Algoritmo no nível externo**: facilita comparar consumo de tempo entre arquiteturas
- **Fase no nível intermediário**: cada GPU finaliza uma fase completa antes de mudar
- **Seed no nível interno**: replicações próximas no tempo (menos variabilidade ambiental)

A função `train_one` é **idempotente**: se cair na execução, é só rodar de novo — tudo que já completou é pulado.

**Estimativa de tempo (GPU mediana, 500k timesteps):**
- DQN: ~2-3h × 12 = ~30h
- PPO: ~1-1.5h × 12 = ~15h
- A2C: ~1-1.5h × 12 = ~15h
- **Total: ~60h** de GPU contínua. Rode em servidor ou parta em lotes via `STAGES_TO_RUN`.

In [ ]:
def run_all_experiments():
    """Executa toda a matriz (algo × stage × seed). Idempotente."""
    total = len(ALGOS) * len(STAGES_TO_RUN) * len(SEEDS_TO_RUN)
    done = 0
    results = []

    for algo in ALGOS:
        for stage in STAGES_TO_RUN:
            for seed in SEEDS_TO_RUN:
                done += 1
                print(f"\n{'='*70}")
                print(f"[{done}/{total}] {algo} | stage {stage} | seed {seed}")
                print('='*70)
                try:
                    csv_path = train_one(algo=algo, stage=stage, seed=seed)
                    results.append((algo, stage, seed, csv_path))
                except Exception as e:
                    print(f"✗ FALHOU [{algo}/{stage}/{seed}]: {e}")
                    results.append((algo, stage, seed, None))

    return results

# Descomente para rodar (cuidado: ~60h no modo completo).
# results = run_all_experiments()
print("Pronto para rodar. Descomente a chamada de `run_all_experiments()` quando estiver pronto.")

## 10. Carregamento e agregação dos logs

Lê todos os CSVs gerados, anexa identificadores de configuração e consolida em um único `DataFrame` longo — formato ideal para análise estatística e plotagem.

In [ ]:
def load_all_logs() -> pd.DataFrame:
    """Concatena todos os CSVs de avaliação em um DataFrame longo."""
    rows = []
    for csv_path in sorted(LOGS_DIR.glob("*.csv")):
        name = csv_path.stem  # ex: PPO_stage1-1_seed42
        try:
            algo, stage_part, seed_part = name.split("_")
            stage = stage_part.replace("stage", "")
            seed = int(seed_part.replace("seed", ""))
        except ValueError:
            print(f"  ⚠ ignorando {csv_path.name} (nome fora do padrão)")
            continue

        df = pd.read_csv(csv_path)
        df["algo"] = algo
        df["stage"] = stage
        df["seed"] = seed
        rows.append(df)

    if not rows:
        print("Nenhum log encontrado.")
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


df_all = load_all_logs()
print(f"Total de avaliações registradas: {len(df_all):,}")
if len(df_all) > 0:
    print(f"Configurações distintas: {df_all.groupby(['algo','stage','seed']).ngroups}")
    print(f"\nColunas: {list(df_all.columns)}")
    print(f"\nAmostra:")
    display(df_all.head())

## 11. Métricas do Grupo I — comparação entre arquiteturas

Para cada `(algo, stage, seed)` calculamos:

- **$\bar{R}_{\text{final}}$**: média da recompensa nos últimos 50k timesteps (regime estacionário)
- **AUC**: integral normalizada da curva de aprendizado (sample efficiency)
- **$t_{80\%}$**: timesteps até atingir 80% da recompensa máxima observada
- **$\sigma_{\text{final}}$**: desvio-padrão da recompensa nos últimos 50k timesteps (estabilidade)

Depois agregamos por `(algo, stage)` reportando **mediana ± IQR** sobre as 3 seeds — robusto a outliers, padrão em RL.

In [ ]:
def compute_group1_metrics(df_all: pd.DataFrame, final_window: int = 50_000) -> pd.DataFrame:
    """
    Calcula métricas do Grupo I para cada (algo, stage, seed).
    """
    if df_all.empty:
        return pd.DataFrame()

    rows = []
    for (algo, stage, seed), g in df_all.groupby(["algo", "stage", "seed"]):
        g = g.sort_values("timestep")
        # Recompensa média por timestep de avaliação
        per_t = g.groupby("timestep")["reward"].mean().reset_index()
        per_t = per_t.sort_values("timestep")

        max_t = per_t["timestep"].max()
        max_r = per_t["reward"].max()

        # R_final: média dos últimos 50k
        final_mask = per_t["timestep"] >= (max_t - final_window)
        R_final = per_t.loc[final_mask, "reward"].mean()
        sigma_final = per_t.loc[final_mask, "reward"].std()

        # AUC normalizada (trapezoidal / max_t)
        auc = np.trapz(per_t["reward"], per_t["timestep"]) / max(max_t, 1)

        # t_80%: primeiro timestep onde reward >= 0.8 * max
        threshold = 0.8 * max_r
        above = per_t[per_t["reward"] >= threshold]
        t80 = int(above["timestep"].iloc[0]) if len(above) > 0 else np.nan

        rows.append(dict(
            algo=algo, stage=stage, seed=seed,
            R_final=R_final, AUC=auc, t80=t80, sigma_final=sigma_final,
        ))

    return pd.DataFrame(rows)


def aggregate_group1(metrics_g1: pd.DataFrame) -> pd.DataFrame:
    """Agrega Grupo I por (algo, stage): mediana, IQR."""
    agg = (metrics_g1
        .groupby(["algo", "stage"])
        .agg(R_final_median=("R_final", "median"),
             R_final_iqr=("R_final", lambda x: x.quantile(0.75) - x.quantile(0.25)),
             AUC_median=("AUC", "median"),
             AUC_iqr=("AUC", lambda x: x.quantile(0.75) - x.quantile(0.25)),
             t80_median=("t80", "median"),
             sigma_final_median=("sigma_final", "median"))
        .reset_index())
    return agg


if len(df_all) > 0:
    metrics_g1 = compute_group1_metrics(df_all)
    metrics_g1.to_csv(METRICS_DIR / "group1_per_seed.csv", index=False)

    metrics_g1_agg = aggregate_group1(metrics_g1)
    metrics_g1_agg.to_csv(METRICS_DIR / "group1_aggregated.csv", index=False)

    print("=== Grupo I — agregado por (algo, stage) ===\n")
    display(metrics_g1_agg.round(2))

## 12. Métricas do Grupo II — avaliação automática de dificuldade

Calculadas sobre os episódios da **última avaliação** de cada treino (regime final do agente):

- **$\tau$ = taxa de conclusão** (média de `flag_get`)
- **$\bar{d}$ = distância normalizada** (max_x_pos / comprimento da fase) — comprimentos canônicos extraídos da documentação do `gym-super-mario-bros`
- **mortes médias** por episódio
- **tempo médio até conclusão** (apenas episódios bem-sucedidos)

In [ ]:
# Comprimentos canônicos das fases (em unidades de x_pos do emulador NES)
# Fontes: documentação do gym-super-mario-bros e Mario AI Benchmark
STAGE_LENGTH = {
    "1-1": 3266,
    "1-2": 3266,
    "4-1": 3866,
    "8-1": 3266,
}


def compute_group2_metrics(df_all: pd.DataFrame) -> pd.DataFrame:
    """
    Grupo II: métricas de dificuldade sobre a ÚLTIMA avaliação de cada treino.
    """
    if df_all.empty:
        return pd.DataFrame()

    rows = []
    for (algo, stage, seed), g in df_all.groupby(["algo", "stage", "seed"]):
        last_t = g["timestep"].max()
        last_eval = g[g["timestep"] == last_t]

        tau = last_eval["flag_get"].mean()
        d_bar = (last_eval["max_x_pos"] / STAGE_LENGTH.get(stage, 3266)).mean()
        deaths_mean = last_eval["deaths"].mean()

        successful = last_eval[last_eval["flag_get"] == 1]
        time_mean = successful["frames"].mean() if len(successful) > 0 else np.nan

        rows.append(dict(
            algo=algo, stage=stage, seed=seed,
            tau=tau, d_bar=d_bar, deaths=deaths_mean, time_frames=time_mean,
        ))

    return pd.DataFrame(rows)


def aggregate_group2(metrics_g2: pd.DataFrame) -> pd.DataFrame:
    return (metrics_g2
        .groupby(["algo", "stage"])
        .agg(tau_median=("tau", "median"),
             tau_iqr=("tau", lambda x: x.quantile(0.75) - x.quantile(0.25)),
             d_bar_median=("d_bar", "median"),
             deaths_median=("deaths", "median"),
             time_median=("time_frames", "median"))
        .reset_index())


if len(df_all) > 0:
    metrics_g2 = compute_group2_metrics(df_all)
    metrics_g2.to_csv(METRICS_DIR / "group2_per_seed.csv", index=False)

    metrics_g2_agg = aggregate_group2(metrics_g2)
    metrics_g2_agg.to_csv(METRICS_DIR / "group2_aggregated.csv", index=False)

    print("=== Grupo II — agregado por (algo, stage) ===\n")
    display(metrics_g2_agg.round(3))

## 13. Análise estatística — Spearman entre ranking dos agentes e dificuldade canônica

Para cada algoritmo, calcula a correlação de Spearman entre:

- O **ranking das fases pela métrica do agente** ($\tau$, $\bar{d}$, mortes, tempo)
- O **ranking canônico de dificuldade** das fases (1-1 → 1-2 → 4-1 → 8-1, ordem 1 a 4)

**Hipóteses:**
- $\tau$ e $\bar{d}$: esperamos $\rho \to -1$ (quanto mais difícil, menor a métrica)
- mortes: esperamos $\rho \to +1$ (quanto mais difícil, mais mortes)

Com $n=4$ fases o poder estatístico é limitado — registramos $\rho$ e $p$-valor mas o foco é a **direção e magnitude** da correlação como evidência exploratória.

In [ ]:
def spearman_difficulty(metrics_g2_agg: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada algoritmo, correlaciona métricas com o ranking canônico das fases.
    """
    if metrics_g2_agg.empty:
        return pd.DataFrame()

    rank_canonical = {s: i for i, s in enumerate(STAGES, start=1)}

    rows = []
    for algo in metrics_g2_agg["algo"].unique():
        sub = metrics_g2_agg[metrics_g2_agg["algo"] == algo].copy()
        sub["rank_canonical"] = sub["stage"].map(rank_canonical)
        sub = sub.sort_values("rank_canonical")

        if len(sub) < 3:
            continue

        for metric, expected_sign in [
            ("tau_median",    "-"),
            ("d_bar_median",  "-"),
            ("deaths_median", "+"),
            ("time_median",   "+"),
        ]:
            x = sub["rank_canonical"].values
            y = sub[metric].values
            if np.all(np.isnan(y)):
                continue
            rho, p = spearmanr(x, y, nan_policy="omit")
            rows.append(dict(
                algo=algo, metric=metric, expected_sign=expected_sign,
                rho=rho, p_value=p, n=len(sub),
            ))

    return pd.DataFrame(rows)


if len(df_all) > 0:
    spearman_df = spearman_difficulty(metrics_g2_agg)
    spearman_df.to_csv(METRICS_DIR / "spearman_difficulty.csv", index=False)

    print("=== Spearman: ranking do agente vs dificuldade canônica ===\n")
    display(spearman_df.round(3))

## 14. Análise estatística — Mann-Whitney U entre algoritmos

Teste não-paramétrico pareado **por fase** comparando todos os pares de algoritmos (DQN vs PPO, DQN vs A2C, PPO vs A2C) na métrica de recompensa final. Escolha justificada pelo pequeno $n$ amostral (3 seeds) e não-normalidade típica em RL.

In [ ]:
from itertools import combinations

def mannwhitney_between_algos(metrics_g1: pd.DataFrame, metric: str = "R_final") -> pd.DataFrame:
    if metrics_g1.empty:
        return pd.DataFrame()
    rows = []
    for stage in metrics_g1["stage"].unique():
        sub = metrics_g1[metrics_g1["stage"] == stage]
        for a, b in combinations(sorted(sub["algo"].unique()), 2):
            xa = sub[sub["algo"] == a][metric].dropna().values
            xb = sub[sub["algo"] == b][metric].dropna().values
            if len(xa) < 2 or len(xb) < 2:
                continue
            stat, p = mannwhitneyu(xa, xb, alternative="two-sided")
            rows.append(dict(
                stage=stage, comparison=f"{a} vs {b}",
                median_diff=float(np.median(xa) - np.median(xb)),
                U=stat, p_value=p, significant=bool(p < 0.05),
            ))
    return pd.DataFrame(rows)


if len(df_all) > 0:
    mw_df = mannwhitney_between_algos(metrics_g1, metric="R_final")
    mw_df.to_csv(METRICS_DIR / "mannwhitney_algos.csv", index=False)

    print("=== Mann-Whitney U entre algoritmos (R_final, por fase) ===\n")
    display(mw_df.round(4))

## 15. Plots — Curvas de aprendizado

Para cada fase, plotamos a média ± banda IQR da recompensa por algoritmo, ao longo dos timesteps. Esse é o gráfico "carro-chefe" da Seção de Resultados do artigo.

In [ ]:
def plot_learning_curves(df_all: pd.DataFrame, savedir: Path = PLOTS_DIR):
    if df_all.empty:
        print("Sem dados para plotar.")
        return

    stages_present = sorted(df_all["stage"].unique())
    n = len(stages_present)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=False)
    if n == 1:
        axes = [axes]

    palette = {"DQN": "#1f77b4", "PPO": "#ff7f0e", "A2C": "#2ca02c"}

    for ax, stage in zip(axes, stages_present):
        sub = df_all[df_all["stage"] == stage]
        for algo, g in sub.groupby("algo"):
            agg = (g.groupby("timestep")["reward"]
                   .agg(["median",
                         lambda x: x.quantile(0.25),
                         lambda x: x.quantile(0.75)])
                   .rename(columns={"<lambda_0>": "q25", "<lambda_1>": "q75"})
                   .reset_index())
            color = palette.get(algo, None)
            ax.plot(agg["timestep"], agg["median"], label=algo, color=color, linewidth=2)
            ax.fill_between(agg["timestep"], agg["q25"], agg["q75"], alpha=0.2, color=color)
        ax.set_title(f"Fase {stage}")
        ax.set_xlabel("Timesteps")
        ax.set_ylabel("Recompensa (mediana ± IQR)")
        ax.legend(loc="best", fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    out = savedir / "learning_curves.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Salvo em {out}")


if len(df_all) > 0:
    plot_learning_curves(df_all)

## 16. Plot — Comparação Grupo I (DQN vs PPO vs A2C)

Bar charts agregando $\bar{R}_{\text{final}}$, AUC e $t_{80\%}$ por algoritmo × fase.

In [ ]:
def plot_group1_comparison(metrics_g1: pd.DataFrame, savedir: Path = PLOTS_DIR):
    if metrics_g1.empty:
        print("Sem dados.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    palette = {"DQN": "#1f77b4", "PPO": "#ff7f0e", "A2C": "#2ca02c"}

    for ax, metric, title in zip(
        axes,
        ["R_final", "AUC", "t80"],
        ["Recompensa final média", "AUC normalizada", "Timesteps até 80% do máximo"],
    ):
        sns.barplot(data=metrics_g1, x="stage", y=metric, hue="algo",
                    palette=palette, ax=ax, errorbar=("ci", 95),
                    order=sorted(metrics_g1["stage"].unique()))
        ax.set_title(title)
        ax.set_xlabel("Fase")
        ax.set_ylabel(metric)
        ax.legend(title="Algoritmo")

    plt.tight_layout()
    out = savedir / "group1_comparison.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Salvo em {out}")


if len(df_all) > 0:
    plot_group1_comparison(metrics_g1)

## 17. Plot — Grupo II: métricas vs ordem canônica de dificuldade

Visualiza como cada métrica (taxa de conclusão, distância normalizada, mortes) varia ao longo das fases na ordem canônica. Idealmente queremos ver $\tau$ e $\bar{d}$ decrescentes e mortes crescentes.

In [ ]:
def plot_group2_difficulty(metrics_g2: pd.DataFrame, savedir: Path = PLOTS_DIR):
    if metrics_g2.empty:
        print("Sem dados.")
        return

    metric_titles = {
        "tau":     "Taxa de conclusão (τ)",
        "d_bar":   "Distância normalizada (d̄)",
        "deaths":  "Mortes por episódio",
    }
    palette = {"DQN": "#1f77b4", "PPO": "#ff7f0e", "A2C": "#2ca02c"}
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    stage_order = [s for s in STAGES if s in metrics_g2["stage"].unique()]

    for ax, (metric, title) in zip(axes, metric_titles.items()):
        for algo, g in metrics_g2.groupby("algo"):
            agg = g.groupby("stage")[metric].agg(["median",
                                                   lambda x: x.quantile(0.25),
                                                   lambda x: x.quantile(0.75)])
            agg.columns = ["median", "q25", "q75"]
            agg = agg.reindex(stage_order)
            x = np.arange(len(stage_order))
            ax.plot(x, agg["median"], "o-", label=algo, linewidth=2,
                    markersize=8, color=palette.get(algo))
            ax.fill_between(x, agg["q25"], agg["q75"], alpha=0.2,
                            color=palette.get(algo))
        ax.set_xticks(x)
        ax.set_xticklabels(stage_order)
        ax.set_xlabel("Fase (ordem canônica de dificuldade →)")
        ax.set_ylabel(title)
        ax.set_title(title)
        ax.legend(title="Algoritmo")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    out = savedir / "group2_difficulty.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Salvo em {out}")


if len(df_all) > 0:
    plot_group2_difficulty(metrics_g2)

## 18. Resumo final e próximos passos

Ao final desta execução, os seguintes artefatos foram gerados em `./mario_drl_results/`:

```
mario_drl_results/
├── models/       # Modelos finais (.zip do SB3) — um por experimento
├── logs/         # CSVs com métricas por episódio de avaliação
├── tensorboard/  # Logs do TensorBoard para inspeção das curvas
├── metrics/      # CSVs consolidados (Grupos I, II, Spearman, Mann-Whitney)
└── plots/        # PNGs para inserir no artigo
```

**Próximos passos previstos:**

1. **Executar o experimento completo**: setar `SMOKE_TEST=False` e rodar `run_all_experiments()`.
2. **Refatorar para estrutura modular**: scripts CLI (`train.py`, `analyze.py`), configs YAML, fácil de reproduzir e versionar (próxima fase do projeto).
3. **Incorporar resultados ao artigo .tex**: as tabelas dos CSVs em `metrics/` e os PNGs de `plots/` vão direto para a Seção 3 (Experimentos e Resultados) do .tex que já está escrito.
4. **Discussão crítica**: complementar a Seção 4 (Conclusões) com a interpretação dos sinais de $\rho$ encontrados e as diferenças significativas no Mann-Whitney.

Para inspecionar curvas de aprendizado interativas:

```bash
tensorboard --logdir mario_drl_results/tensorboard
```